In [ ]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


## importation de model et base de donnée

In [ ]:
from src.preprocessing import preprocess_full_dataset
from src.dataset import create_dataloaders
from src.models import HybridCNNBiLSTMAttention

# Exécuter le prétraitement
df_clean, recommended_length = preprocess_full_dataset(r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv')
# Créer les dataloaders PyTorch
train_loader, test_loader, vocab, train_df, test_df = create_dataloaders(
    df_clean,
    max_length=60,
    batch_size=64
)

# Initialisation du modèle hybride
model = HybridCNNBiLSTMAttention(
    vocab_size=len(vocab),
    embedding_dim=256,
    n_filters=64,          # ↓ de 128 à 64 pour équilibrer les branches
    filter_sizes=[3, 4, 5],
    hidden_dim=128,
    num_layers=2,
    output_dim=3,
    dropout=0.5
)

print(f"🧠 Modèle Hybride CNN + BiLSTM + Attention initialisé :")
print(f"   - Embedding dim     : 256")
print(f"   - CNN Filtres       : 64 × 3 tailles (3,4,5)")
print(f"   - BiLSTM Hidden dim : 128 (×2 bidirectionnel)")
print(f"   - Fusion            : Parallèle (CNN features + LSTM context)")
print(f"   - Paramètres        : {sum(p.numel() for p in model.parameters()):,}")

🚀 PRÉTRAITEMENT DU DATASET

✅ Dataset chargé : 31,232 échantillons
🧹 Prétraitement en cours...
✅ 31,202 échantillons conservés

📏 Longueur recommandée (95e percentile) : 56 tokens

📚 Vocabulaire construit : 11703 mots uniques

📦 Dataloaders créés :
   - Train : 391 batches
   - Test  : 98 batches
🧠 Modèle Hybride CNN + BiLSTM + Attention initialisé :
   - Embedding dim     : 256
   - CNN Filtres       : 64 × 3 tailles (3,4,5)
   - BiLSTM Hidden dim : 128 (×2 bidirectionnel)
   - Fusion            : Parallèle (CNN features + LSTM context)
   - Paramètres        : 4,017,668


## l'entrainement

In [ ]:
from src.trainer import train_model
import torch
import torch.nn as nn

# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Entraînement avec monitoring complet
history, model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=30,
    patience=4,
    save_dir=r"C:\Users\SOL\Downloads\sentiment_analysis_project\models\trainary",
    model_name="hybrid",
    verbose=True,
)



🚀 DÉBUT DE L'ENTRAÎNEMENT : hybrid
Epoch  | Train Loss   Train F1   Val Loss     Val F1     Val Acc    | Time   | Status      
--------------------------------------------------------------------------------
     1 | 1.4304       0.4446     0.8952       0.5485     0.5824     | 5     s | ★ BEST      
     2 | 0.9033       0.5761     0.8132       0.6381     0.6441     | 5     s | ★ BEST      
     3 | 0.8338       0.6221     0.7904       0.6521     0.6587     | 5     s | ★ BEST      
     4 | 0.7932       0.6452     0.7663       0.6787     0.6778     | 5     s | ★ BEST      
     5 | 0.7538       0.6680     0.7213       0.6895     0.6877     | 5     s | ★ BEST      
     6 | 0.7232       0.6849     0.7611       0.6949     0.6960     | 5     s | ★ BEST      
     7 | 0.7073       0.6963     0.7438       0.7001     0.7000     | 5     s | ★ BEST      
     8 | 0.6772       0.7092     0.7411       0.7002     0.6999     | 5     s | ★ BEST      
     9 | 0.6538       0.7246     0.7213       0

## évaluation

In [ ]:
# Évaluation finale avec visualisations publication
import torch
from src.evaluate import evaluate_model
# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir="reports/figures",
    model_name="hybrid"
)


📊 ÉVALUATION DU MODÈLE : hybrid

Accuracy      : 0.7042
F1 Macro      : 0.7053
F1 Weighted   : 0.7034
ROC AUC Micro : 0.8699
ROC AUC Macro : 0.8657

Classification Report:
              precision    recall  f1-score   support

     Négatif     0.7040    0.6912    0.6975      1820
      Neutre     0.6548    0.6382    0.6464      2327
     Positif     0.7557    0.7889    0.7720      2094

    accuracy                         0.7042      6241
   macro avg     0.7048    0.7061    0.7053      6241
weighted avg     0.7030    0.7042    0.7034      6241

   → Matrice de confusion sauvegardée : confusion_matrix.png/csv
   → Courbes ROC sauvegardées : roc_curves.png

✅ Évaluation terminée - Résultats sauvegardés dans : reports\figures\hybrid
